Model Context Protocol (MCP) is an open protocol that standardizes how AI models communicate with external tools, data sources, and applications, enabling seamless tool discovery and interaction.

          ┌───────────────┐
          │   AI Agent    │  (Client)
          └───────┬───────┘
                  │
                  │  "What tools do you have?"
                  ▼
          ┌───────────────┐
          │ MCP Server    │
          │ (Tool Catalog)│
          └───────┬───────┘
                  │
                  │  Returns list of tools
                  ▼
        ┌───────────────────────────┐
        │ Tool Discovery Protocol   │
        │ e.g., search_hotels,      │
        │ list_hotels, book_hotel   │
        └───────────┬───────────────┘
                    │
                    │ Agent chooses tool
                    ▼
        ┌───────────────────────────┐
        │ Tool Execution            │
        │ MCP Server calls APIs:    │
        │ Expedia, Booking.com, OYO │
        └───────────┬───────────────┘
                    │
                    │ Results merged
                    ▼
          ┌───────────────┐
          │   AI Agent    │
          │ Shows results │
          └───────────────┘


Client (Agent): The AI that interprets user queries. Example: your client.py.

Server: The place where tools live. Example: your Flask server exposing search_hotels, list_hotels, book_hotel.

The client asks: “What tools do you provide?”

The server replies with a catalog of tools (like a menu).

The client then calls the right tool with the right inputs.

Tool Discovery Protocol
Instead of hardcoding tools, the agent uses MCP’s discovery step:

Client calls /tools.

Server responds with a list of tools + their input/output schemas.

Agent now knows what’s available and how to use them.

In [15]:
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import git
from mcp import ClientSession, StdioServerParameters, stdio_client
from mcp.types import CallToolResult, ListToolsResult


In [16]:
# Cell 2: Utility functions

def format_tool_result(result: CallToolResult) -> str:
    """Format MCP tool result into readable text."""
    if result.content:
        texts = []
        for block in result.content:
            text = getattr(block, "text", None)
            if text:
                texts.append(text.strip())
        if texts:
            return "\n".join(texts)

    if result.structuredContent is not None:
        return json.dumps(result.structuredContent, indent=2)

    return "<no result>"

def summarize_tool_text(text: str) -> str:
    """Summarize tool output for comparison."""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if not lines:
        return "<no result>"
    if lines[0].startswith("Repository status:") and len(lines) > 1:
        return f"{lines[0]} {lines[1]}"
    return lines[0]


In [17]:
# Cell 3: Direct Git functions using GitPython

def direct_git_status(repo_path: Path) -> str:
    repo = git.Repo(repo_path)
    return repo.git.status()

def direct_git_branch(repo_path: Path, branch_type: str = "local") -> str:
    repo = git.Repo(repo_path)
    if branch_type == "local":
        return repo.git.branch()
    if branch_type == "remote":
        return repo.git.branch("-r")
    return repo.git.branch("-a")

def direct_git_log(repo_path: Path, max_count: int = 1) -> str:
    repo = git.Repo(repo_path)
    lines = []
    for commit in repo.iter_commits(max_count=max_count):
        lines.append(
            f"Commit: {commit.hexsha}\n"
            f"Author: {commit.author}\n"
            f"Date: {commit.authored_datetime}\n"
            f"Message: {commit.message.strip()}"
        )
    return "\n\n".join(lines)

def direct_git_show(repo_path: Path, revision: str = "HEAD") -> str:
    repo = git.Repo(repo_path)
    commit = repo.commit(revision)
    output = [
        f"Commit: {commit.hexsha}",
        f"Author: {commit.author}",
        f"Date: {commit.authored_datetime}",
        f"Message: {commit.message.strip()}",
        "--- diff ---",
    ]
    diff = commit.diff(commit.parents[0] if commit.parents else git.NULL_TREE, create_patch=True)
    for d in diff:
        output.append(f"{d.a_path} -> {d.b_path}")
        if d.diff is not None:
            output.append(d.diff.decode("utf-8", errors="ignore") if isinstance(d.diff, bytes) else str(d.diff))
    return "\n".join(output)


In [18]:
# Cell 4: Agent flow with comparison

async def run_client_with_comparison(repo_path: Path):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["-m", "mcp_server_git", "--repository", str(repo_path)],
        cwd=str(repo_path),
    )

    print("Starting MCP Git server...\n")

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            # === Agent asks: "What are your tools?" ===
            tools = await session.list_tools()
            print("Agent: What are your tools?")
            print("Server replied with tools:\n")
            for tool in tools.tools:
                print(f"- {tool.name}: {tool.description}")

            # === Comparison tests ===
            print("\n=== Comparison summary ===\n")
            tool_tests = [
                ("git_status", {"repo_path": str(repo_path)}, direct_git_status),
                ("git_branch", {"repo_path": str(repo_path), "branch_type": "local"}, lambda rp: direct_git_branch(rp, "local")),
                ("git_log", {"repo_path": str(repo_path), "max_count": 1}, lambda rp: direct_git_log(rp, 1)),
                ("git_show", {"repo_path": str(repo_path), "revision": "HEAD"}, lambda rp: direct_git_show(rp, "HEAD")),
            ]

            for tool_name, payload, direct_fn in tool_tests:
                mcp_result = await session.call_tool(tool_name, payload)
                mcp_text = summarize_tool_text(format_tool_result(mcp_result))
                direct_text = summarize_tool_text(direct_fn(repo_path))
                same = "yes" if direct_text in mcp_text or mcp_text in direct_text else "no"

                print(f"Tool: {tool_name}")
                print(f"  MCP output:   {mcp_text}")
                print(f"  Direct output: {direct_text}")
                print(f"  Match: {same}\n")


In [20]:
# Cell 5: Run the comparison in Jupyter

repo_path = Path(".").resolve()  # current folder as repo
if not (repo_path / ".git").exists():
    raise SystemExit(f"{repo_path} is not a Git repository")

# In Jupyter, just await the coroutine directly
await run_client_with_comparison(repo_path)


Starting MCP Git server...

Agent: What are your tools?
Server replied with tools:

- git_status: Shows the working tree status
- git_diff_unstaged: Shows changes in the working directory that are not yet staged
- git_diff_staged: Shows changes that are staged for commit
- git_diff: Shows differences between branches or commits
- git_commit: Records changes to the repository
- git_add: Adds file contents to the staging area
- git_reset: Unstages all staged changes
- git_log: Shows the commit logs
- git_create_branch: Creates a new branch from an optional base branch
- git_checkout: Switches branches
- git_show: Shows the contents of a commit
- git_branch: List Git branches

=== Comparison summary ===

Tool: git_status
  MCP output:   Repository status: On branch master
  Direct output: On branch master
  Match: yes

Tool: git_branch
  MCP output:   * master
  Direct output: * master
  Match: yes

Tool: git_log
  MCP output:   Commit history:
  Direct output: Commit: 455c04ff843d9b501b4

User question → LLM → decides tool + arguments → MCP server → Git repo


In [21]:
import os
import boto3
from dotenv import load_dotenv
from langchain_aws import BedrockLLM

# Load .env file
load_dotenv("myenv.env")

# Read values
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

# Initialize Bedrock client with credentials
bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY
)


In [32]:
import boto3, json

def ask_bedrock_llm(question: str, tools: list[str], 
                    model_id="amazon.nova-micro-v1:0", region="us-east-1"):
    """
    Ask Bedrock Claude LLM which MCP tool to call.
    Returns (tool_name, payload_dict).
    """
    bedrock_runtime = boto3.client("bedrock-runtime", region_name=region)

    prompt = f"""
    You are an AI agent. The user asked: "{question}".
Available tools: {tools}.
IMPORTANT: Always include "repo_path": "{str(Path('.').resolve())}" in the payload.
Respond ONLY with JSON: {{"tool_name": "...", "payload": {{...}}}}
    """

    body = {
        "messages": [
            {"role": "user", "content": [{"text": prompt}]}
        ],
        "inferenceConfig": {"maxTokens": 256, "temperature": 0.2}
    }

    response = bedrock_runtime.invoke_model(
        modelId=model_id,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )

    response = json.loads(response["body"].read())
    text = response["output"]["message"]["content"][0]["text"]

    try:
        parsed = json.loads(text)
        return parsed.get("tool_name"), parsed.get("payload")
    except Exception:
        print("LLM output could not be parsed:", text)
        return None, None


AGENT WITH MCP+LLM

In [33]:
async def run_llm_agent(question: str, repo_path: Path):
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["-m", "mcp_server_git", "--repository", str(repo_path)],
        cwd=str(repo_path),
    )

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            # Agent asks: "What are your tools?"
            tools_result = await session.list_tools()
            tools = [t.name for t in tools_result.tools]
            print("Available tools:", tools)

            # Ask Bedrock LLM which tool to call
            tool_name, payload = ask_bedrock_llm(question, tools)
            if tool_name:
                print(f"\nLLM decided to call: {tool_name} with {payload}")
                result = await session.call_tool(tool_name, payload)
                print("\nTool result:\n")
                print(result)
            else:
                print("LLM could not decide a tool.")


In [35]:
repo_path = Path(".").resolve()
if not (repo_path / ".git").exists():
    raise SystemExit(f"{repo_path} is not a Git repository")

question = "Can you show me the latest commit?"
await run_llm_agent(question, repo_path)


Available tools: ['git_status', 'git_diff_unstaged', 'git_diff_staged', 'git_diff', 'git_commit', 'git_add', 'git_reset', 'git_log', 'git_create_branch', 'git_checkout', 'git_show', 'git_branch']

LLM decided to call: git_log with {'repo_path': '/mnt/c/Users/lahar/OneDrive/Pictures/Desktop/bootcamp'}

Tool result:

meta=None content=[TextContent(type='text', text='Commit history:\nCommit: \'455c04ff843d9b501b4139ddb865a30dd4b66609\'\nAuthor: <git.Actor "lahari-006 <lahari@bosonanalytics.com>">\nDate: 2026-06-22 14:00:21+00:00\nMessage: \'added day14\\n\'\n\nCommit: \'57c89ddb394a5c6f072a31bf333947dbfc3f4ab2\'\nAuthor: <git.Actor "lahari-006 <lahari@bosonanalytics.com>">\nDate: 2026-06-19 13:57:13+00:00\nMessage: \'day13 completed\\n\'\n\nCommit: \'314dcb8ec45dc31047505dc7454f8958f9e98e12\'\nAuthor: <git.Actor "lahari-006 <lahari@bosonanalytics.com>">\nDate: 2026-06-17 13:24:48+00:00\nMessage: \'added day11\\n\'\n\nCommit: \'c2039fe0b1fbd16d446f03e4dfb37418804dd4b5\'\nAuthor: <git.Actor